In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy
from snf.farfield import load_far_field, ff_to_T, nf_from_T, T_to_R, theta_weights, Z0
from snf.fields import load_field, grid_axes
from snf.swe import j_index

DATA = '../data/'

# Define Setup

In [ ]:
freq = 15 * 10**9
k = 2 * np.pi * freq / scipy.constants.c
R_PROBE = 0.4 # Distance from AUT origin to probe far field origin (m)
r0 = 0.095 # AUT minimum sphere radius (m)
S11_probe_dB = -0.999930 # Probe S11 at 15 GHz
P_in = 1.0 # Incident power at each port (W)

N_AUT = 40 # ~ k r0 + 10
N_PROBE = 12
print(f'k r0 = {k * r0:.1f}')

# Far Field -> SWEC

In [ ]:
# T for each antenna/port. Conjugate HFSS (e^{+jwt}) -> Hansen (e^{-iwt})

ff, T = {}, {}
for name, N in (('AUT_wp1', N_AUT), ('AUT_wp2', N_AUT), ('probe_wp1', N_PROBE), ('probe_wp2', N_PROBE)):
    ff[name] = load_far_field(DATA + f'FF_fields/FF_{name}.csv')
    T[name] = ff_to_T(*ff[name], N, P_in=P_in)

# Power check: radiated from far field vs P_in sum |T|^2 vs mismatch
for name, (th, ph, Eth, Eph) in ff.items():
    P_ff = theta_weights(th) @ np.mean(np.abs(Eth)**2 + np.abs(Eph)**2, axis=1) * 2 * np.pi / (2 * Z0)
    print(f'{name}: far field {P_ff:.4f} W, P_in sum|T|^2 {P_in * np.sum(np.abs(T[name])**2):.4f} W')
print(f'Probe P_in (1 - |S11|^2) = {P_in * (1 - 10**(S11_probe_dB / 10)):.4f} W')

# Truncation

In [ ]:
# Power per polar index n (computed with a high N to see the roll-off)

fig, ax = plt.subplots(figsize=(7, 4.5))
N_hi = 85
for name in ff:
    T_hi = ff_to_T(*ff[name], N_hi, P_in=P_in)
    pn = [np.sum(np.abs(T_hi[j_index(np.array([1, 2])[:, None], np.arange(-n, n + 1)[None], n)])**2) for n in range(1, N_hi + 1)]
    ax.plot(np.arange(1, N_hi + 1), 10 * np.log10(pn / np.max(pn)), label=name)
ax.axvline(k * r0, color='gray', ls='--', label='k r0')
ax.set(xlabel='n', ylabel='Relative power (dB)', title='SWE power spectrum', ylim=(-200, 5))
ax.grid(True); ax.legend()

# AUT Near Field Cross-Check

In [ ]:
xyz, E_aut = load_field(DATA + 'SNF_fields_ports/AUT/wp1/E.csv')
_, H_aut = load_field(DATA + 'SNF_fields_ports/AUT/wp1/H.csv')
th_1d, ph_1d, R_sphere = grid_axes(xyz)

sub = np.zeros((th_1d.size, ph_1d.size), bool) # every 4th point to keep this quick
sub[::4, ::4] = True
sub = sub.flatten()
n_hat = xyz[sub] / R_sphere
tan = lambda F: F - np.sum(F * n_hat, axis=1)[:, None] * n_hat
err = lambda a, b: np.linalg.norm(a - b) / np.linalg.norm(b)

for conj in (True, False):
    T_c = ff_to_T(*ff['AUT_wp1'], N_AUT, P_in=P_in, conj=conj)
    E_swe, H_swe = nf_from_T(T_c, xyz[sub], k, P_in=P_in)
    E_ref, H_ref = (E_aut[sub].conj(), H_aut[sub].conj()) if conj else (E_aut[sub], H_aut[sub])
    print(f'conj={conj}: E {err(E_swe, E_ref):.4f} (tangential {err(tan(E_swe), tan(E_ref)):.4f}), H {err(H_swe, H_ref):.4f}')

In [ ]:
# Error vs N

for N in (10, 20, 25, 30, 40):
    E_swe, H_swe = nf_from_T(ff_to_T(*ff['AUT_wp1'], N, P_in=P_in), xyz[sub], k, P_in=P_in)
    print(f'N={N}: E tangential {err(tan(E_swe), tan(E_aut[sub].conj())):.4f}, H {err(H_swe, H_aut[sub].conj()):.4f}')

In [ ]:
# E/H-plane cuts of |E tangential|: HFSS vs SWE

nearest = lambda a: np.argmin(np.abs(np.angle(np.exp(1j * (ph_1d - a)))))
grid = np.arange(len(xyz)).reshape(th_1d.size, ph_1d.size)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, (a, b, plane) in zip(axes, ((0, np.pi, 'E'), (np.pi / 2, 3 * np.pi / 2, 'H'))):
    idx = np.r_[grid[::-1, nearest(b)], grid[:, nearest(a)]]
    theta_cut = np.r_[-np.degrees(th_1d[::-1]), np.degrees(th_1d)]
    E_swe, _ = nf_from_T(T['AUT_wp1'], xyz[idx], k, P_in=P_in)
    n_c = xyz[idx] / R_sphere
    mag = lambda F: np.linalg.norm(F - np.sum(F * n_c, axis=1)[:, None] * n_c, axis=1)
    ref = mag(E_aut[idx].conj())
    ax.plot(theta_cut, 20 * np.log10(ref / ref.max()), 'k-', lw=1.5, label='HFSS')
    ax.plot(theta_cut, 20 * np.log10(mag(E_swe) / ref.max()), 'r--', lw=1.5, label='SWE')
    ax.set(xlabel='Theta (deg)', ylabel='Field strength (dB)', title=f'AUT E at r = {R_sphere:.2f} m, {plane}-plane', xlim=(-180, 180), ylim=(-50, 5))
    ax.grid(True); ax.legend()
plt.tight_layout()

# Probe Near Field Cross-Check

In [ ]:
# Probe sits on +z looking at the origin. Sweep its expansion origin along z to confirm A

_, E_pr = load_field(DATA + 'SNF_fields_ports/probe/wp1/E.csv')
_, H_pr = load_field(DATA + 'SNF_fields_ports/probe/wp1/H.csv')

deltas = np.arange(-3, 3.01, 0.25) * 1e-3
errs = []
for d in deltas:
    E_swe, H_swe = nf_from_T(T['probe_wp1'], xyz[sub], k, origin=np.array([0, 0, R_PROBE + d]), P_in=P_in)
    errs.append(err(H_swe, H_pr[sub].conj()))

plt.figure(figsize=(7, 4.5))
plt.semilogy(deltas * 1e3, errs, 'ko-')
plt.xlabel('Probe origin offset from R_PROBE (mm)')
plt.ylabel('Relative H error')
plt.grid(True)
print(f'Best offset: {deltas[np.argmin(errs)] * 1e3:.2f} mm, error {np.min(errs):.4f}')

# Probe Receive Coefficients

In [ ]:
# Compare n = 1 receive coefficients to the ideal dipole probe, and check how much power is outside |mu| = 1

for name in ('probe_wp1', 'probe_wp2'):
    R = T_to_R(T[name])
    ref = R[j_index(2, -1, 1)]
    print(name, {f'R({s},{m},1)': np.round(R[j_index(s, m, 1)] / ref, 3) for s in (1, 2) for m in (-1, 1)})
    P = np.abs(R)**2
    mu1 = sum(P[j_index(s, m, n)] for n in range(1, N_PROBE + 1) for s in (1, 2) for m in (-1, 1))
    print(f'    power fraction in |mu| = 1: {mu1 / P.sum():.4f}')